<h1 style="color:#D4A017; font-weight:900; font-size:34px;">
  🟨 I. RAG Moderne avec LCEL 
</h1>



<h2 style="color:#FF8C42; font-weight:bold;">
  Étape 1 — Utiliser WordIngestion pour préparer les chunks
</h2>


In [7]:
%pip install langchain_chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 591.3 kB/s eta 0:00:0036m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 15.2 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 20.7 MB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 15.2 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 19.3 MB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.8/141.8 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from word_ingestion import WordIngestion

# ingestor personnalisé 
ingestor = WordIngestion(
    chunk_size= 500,
    chunk_overlap= 80
)

# Traiter un document word 
chunks = ingestor.process("./Document_RAG_Word.docx")

print("Nombre de chunks :", len(chunks))

# Aperçu des premiers chunks 
for i, c in enumerate(chunks[:3]):
    print("=" * 80)
    print(f"CHUNK {i+1}")
    print("=" *80)
    print(c.page_content[:500], "\n")


Nombre de chunks : 7
CHUNK 1
Procédure interne – Ingestion de documents Word pour un pipeline RAG Ce document est conçu spécialement pour accompagner la narration de la section dédiée à l’ingestion des fichiers Word (.docx) dans un pipeline RAG professionnel. Il contient des structures réalistes : titres, sous-titres, listes, tableaux, paragraphes longs et séparations logiques. L’objectif est de démontrer les limites et les forces des loaders Docx2txt et Unstructured dans un contexte réel. 1 

CHUNK 2
. 1. Contexte général Les entreprises produisent un volume important de documents Word au quotidien. Ils contiennent des informations essentielles pour un système RAG, comme des procédures internes, des notes de cadrage, des comptes rendus de réunion et des dossiers techniques. Ces documents doivent être ingérés, nettoyés, découpés en chunks et indexés pour permettre à un modèle de répondre de façon cohérente aux requêtes des utilisateurs. 2 

CHUNK 3
. 2. Challenges rencontrés Voici les 



<h2 style="color:#FF8C42; font-weight:bold;">
  Étape 2 — Créer les embeddings + Vector Store (Chroma)
</h2>


In [33]:
# Import Embeddings
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Initialisation du Vector Store
vector_store = Chroma(
    collection_name="rag_word_docs",
    persist_directory="chroma_word_store",
    embedding_function=embeddings
)

# Ajouter tous mes chunks dans le vector store
vector_store.add_documents(chunks)

print("Index vectoriel initalisé")

Index vectoriel initalisé


<h2 style="color:#FF8C42; font-weight:bold;">
  Étape 3 — Créer le Retriever
</h2>


In [34]:
retriever = vector_store.as_retriever(search_kwargs={"k" : 3})

In [35]:
# afficher un exemple de récupération
retriever.invoke("Quels sont les défis liés à l'extraction du texte Word ?")

[Document(id='584ed883-1d57-40a8-b198-d578d19fe513', metadata={'source': '../data/word_files/Document_RAG_Word.docx'}, page_content=". 2. Challenges rencontrés Voici les défis typiques rencontrés lors de l'extraction de texte Word : - Multiples sauts de ligne - Listes mal reconstruites - Tableaux transformés en texte - Titres non reconnus - Retours chariots inutiles 3. Exemple de contenu complexe Le paragraphe suivant illustre un contenu souvent rencontré dans les entreprises. Il contient des descriptions opérationnelles, des éléments techniques et des blocs narratifs importants pour tester le comportement des extracteurs"),
 Document(id='34f0e9f7-0f3d-4e87-b397-7e87b5b95a50', metadata={'source': '../data/word_files/Document_RAG_Word.docx'}, page_content='. 1. Contexte général Les entreprises produisent un volume important de documents Word au quotidien. Ils contiennent des informations essentielles pour un système RAG, comme des procédures internes, des notes de cadrage, des comptes r

<h2 style="color:#FF8C42; font-weight:bold;">
  Étape 4 — Définir le Prompt + le pipeline LCEL
</h2>


In [36]:
# Import
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_template("""
Tu es un assistant expert des systèmes RAG.
Réponds UNIQUEMENT à partir du contexte fourni ci-dessous.
Si l'information n'est pas présente, réponds simplement : "Ce n'est pas précisé dans le document."

Contexte :                                                                                           
{context}

Question :
{question} 
                                                                                                                                                                                                  
Réponse : 
""")

rag_chain = (
    {"context" : lambda x:retriever.invoke(x["question"]),
     "question" : lambda x: x["question"]}
    | prompt
    | llm
    | StrOutputParser()
)

<h2 style="color:#FF8C42; font-weight:bold;">
  Étape 4 — Execution : Poser une question sur le document Word
</h2>


<h3 style="color:#C0392B; font-weight:bold;">
   Ingestion Word -> nettoyage -> chunking -> indexation Chroma -> retriever -> pipeline LCEL


In [37]:
# 1 ) Définir la question

question = (
    "Quels sont les principaux défis liés à l'extraction de texte "
    "dans un document Word pour un pipeline RAG ?"
)
question

"Quels sont les principaux défis liés à l'extraction de texte dans un document Word pour un pipeline RAG ?"

In [38]:
# 2) Appeler la chaîne LCEL

response = rag_chain.invoke({"question":question})

In [39]:
# Affichage de la réponse 
print("QUESTION :")
print(question)
print("\nRéponse :")
print(response)


QUESTION :
Quels sont les principaux défis liés à l'extraction de texte dans un document Word pour un pipeline RAG ?

Réponse :
Les principaux défis liés à l'extraction de texte dans un document Word pour un pipeline RAG sont les suivants :
- Multiples sauts de ligne
- Listes mal reconstruites
- Tableaux transformés en texte
- Titres non reconnus
- Retours chariots inutiles


<h1 style="color:#D4A017; font-weight:900; font-size:34px;">
  🟨 II. Ingestion Dynamique
</h1>

In [40]:
# 1) Etape 1 : Ingestion du nouveau document
from word_ingestion import WordIngestion

# ingestor personnalisé 
ingestor = WordIngestion(
    chunk_size= 500,
    chunk_overlap= 80
)

# Traiter un document word 
new_chunks = ingestor.process("../data/word_files/Document_RAG_Word_v2.docx")

print("Nombre de chunks :", len(chunks))

# Aperçu des premiers chunks 
for i, c in enumerate(chunks[:3]):
    print("=" * 80)
    print(f"CHUNK {i+1}")
    print("=" *80)
    print(c.page_content[:500], "\n")


Nombre de chunks : 7
CHUNK 1
Procédure interne – Ingestion de documents Word pour un pipeline RAG Ce document est conçu spécialement pour accompagner la narration de la section dédiée à l’ingestion des fichiers Word (.docx) dans un pipeline RAG professionnel. Il contient des structures réalistes : titres, sous-titres, listes, tableaux, paragraphes longs et séparations logiques. L’objectif est de démontrer les limites et les forces des loaders Docx2txt et Unstructured dans un contexte réel. 1 

CHUNK 2
. 1. Contexte général Les entreprises produisent un volume important de documents Word au quotidien. Ils contiennent des informations essentielles pour un système RAG, comme des procédures internes, des notes de cadrage, des comptes rendus de réunion et des dossiers techniques. Ces documents doivent être ingérés, nettoyés, découpés en chunks et indexés pour permettre à un modèle de répondre de façon cohérente aux requêtes des utilisateurs. 2 

CHUNK 3
. 2. Challenges rencontrés Voici les 

In [41]:
# Etape 2 - Enrichir les métadonnées + ajout à Chroma

#1) Ajouter une métadonnée 'source' pour identifier l'origine
for c in new_chunks :
    c.metadata["source"] = "Document_RAG_Word_v2.docx"

#2) Ajouter les nouveaux chunks au vector store existant
vector_store.add_documents(new_chunks)

print("Nouveaux documents ajoutés au vector store")
print(f"Nombre total de documents indexés : {vector_store._collection.count()}")

Nouveaux documents ajoutés au vector store
Nombre total de documents indexés : 14


In [42]:
# Etape 3 : Vérifier que le retriever "voit" les nouveaux documents

# 1 ) Définir la question

question_new = (
    "Quels sont les principaux défis liés à l'extraction de texte "
    "dans un document Word pour un pipeline RAG ?"
)

# 2) Appeler la chaîne LCEL

response = rag_chain.invoke({"question":question_new})

# Affichage de la réponse 
print("QUESTION :")
print(question)
print("\nRéponse :")
print(response)


QUESTION :
Quels sont les principaux défis liés à l'extraction de texte dans un document Word pour un pipeline RAG ?

Réponse :
Les principaux défis liés à l'extraction de texte dans un document Word pour un pipeline RAG sont : 
- Supprimer les espaces inutiles
- Problème d’affichage
- Préparer le pipeline Okandata


<h1 style="color:#D4A017; font-weight:900; font-size:34px;">
  🟨 III. Techniques avancées RAG : Mémoire conversationnelle + Vector Store
</h1>

In [1]:
# =======================================
#  Etape 1 — IMPORTS & FORMATTEUR DOCS
# =======================================

from operator import itemgetter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough,
    RunnableWithMessageHistory
)
from langchain_core.output_parsers import StrOutputParser

from langchain_community.chat_message_histories import ChatMessageHistory

from pdf_ingestion import PDFIngestion

# Fonction obligatoire pour éviter l'erreur TypeError (liste -> texte)
def format_docs(docs) :
    return "\n\n---\n\n".join(d.page_content for d in docs)

format_docs_runnable = RunnableLambda(format_docs)


In [5]:
# =======================================
#  Etape 2 — INGESTION PDF 
# =======================================

ingestor = PDFIngestion(chunk_size=1200, chunk_overlap=150) 

chunks_pdf = ingestor.process("../data/pdf_files/rapport-activite_credit_mutuelle.pdf")

print("Nombre de chunks chargés :", len(chunks_pdf))
print("Aperçu du premier chunk :", chunks_pdf[4].page_content[:500])

Nombre de chunks chargés : 207
Aperçu du premier chunk : Le modèle mutualiste reste-t-il un atout et un levier pour le développement présent et futur du Groupe ? Daniel Baal : Oui et plus que jamais ! Il repose sur des valeurs de proximité, de solidarité et de gouvernance démocratique, qui font notre singularité et notre force face aux défis économiques et sociétaux actuels Dans un environnement en constante évolution, marqué par la digitalisation, la transition écologique et les attentes croissantes des clients en matière de responsabilité, notre mod


In [8]:
# =======================================
#  Etape 3 — INDEXATION DANS CHROMA
# =======================================

embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")

vector_store_pdf = Chroma(
    collection_name="rapport_mutuelle",
    persist_directory= "chroma_pdf_store",
    embedding_function=embeddings
)

# vector_store_pdf.delete_collection()

vector_store_pdf.add_documents(chunks_pdf)

print("Index PDF en base :", vector_store_pdf._collection.count())

Index PDF en base : 207


In [10]:
# =======================================
#  Etape 4 — RETRIEVER
# =======================================

retriever_pdf = vector_store_pdf.as_retriever(search_kwargs = {"k":3})
retriever_pdf

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001DAA9C10A50>, search_kwargs={'k': 3})

In [12]:
# =======================================
#  Etape 5 — PROMPT AVEC HISTORIQUE
# =======================================

prompt_conv = ChatPromptTemplate.from_messages([
    ("system",
    "Tu es un assistant expert qui analyse le rapport d'activité du Crédit Mutuel."
    "Tu t'appuis uniquement sur les extraits fournis. Réponds clairement et en français"),

    MessagesPlaceholder("history"),

    ("human",
    "Voici les extraits pertinents du rapport : \n\n{context}\n\n"
    "Ma question : {question}")
])


In [13]:
# =======================================
#  Etape 6 — CHAÎNE RAG LCEL
# =======================================

llm = ChatOpenAI(model = "gpt-4o-mini")

rag_chain = (
    {
        "question" : itemgetter("question"),
        "context" : itemgetter("question") | retriever_pdf | format_docs_runnable,
        "history" : itemgetter("history"),   
    }
    | prompt_conv
    | llm
    | StrOutputParser()
)


In [14]:
# =======================================
#  Etape 7 — MEMORIRE CONVERSATIONNELLE
# =======================================

store = {}

def get_session_history(session_id : str) : 
    if session_id not in store :
        store[session_id] = ChatMessageHistory()
    return store[session_id]

rag_with_memory = RunnableWithMessageHistory(


    rag_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history"
)



In [15]:
# =======================================
#  Etape 8 — TEST DU RAG CONVERSATIONNELLE
# =======================================

session_id = "mutuelle_session_1"

def ask(question) :
    response = rag_with_memory.invoke(
        {"question": question},
        config = {"configurable":{"session_id" : session_id}}
    )

    print(f"\n ? {question} \n")
    print("Réponse :\n", response)
    print("\n" + "="*80)

ask("Quels sont les grands axes stratégiques du rapport d'activité ?")
ask("Peux-tu résumer ces axes en mettant l'accent sur la transformation digitale ?")
ask("Comment ces axes stratégiques influencent-ils la relation sociétaire ?")
ask("Et résume moi tout cela en une phrase comme si tu parlais à un sociétaire")


 ? Quels osnt les grands axes stratégiques du rapport d'activité ? 

Réponse :
 Les grands axes stratégiques du rapport d'activité du Crédit Mutuel se déclinent comme suit :

1. **Engagement envers les communautés** : Le Crédit Mutuel met en avant son rôle dans le mécénat et le développement des territoires. Il démontre un souci pour l'innovation et la solidarité, avec des initiatives comme le Dividende Sociétal et le Fonds de dotation.

2. **Protection des clients** : Un cadre est établi pour assurer la protection des informations clients. L'institution anticipe et accompagne les situations de vulnérabilité des clients, en prenant des mesures pour éviter la précarité, comme la suppression des formalités médicales pour les clients fidèles.

3. **Conduite des affaires** : Le Crédit Mutuel s'engage à respecter des valeurs coopératives et mutualistes, avec des plans de gestion des risques liés à la fraude, à la corruption et à la conformité.

4. **Renforcement de la compétitivité europée

In [37]:
# =======================================
# 📌 Cellule 8 — TESTS DU RAG
# =======================================

session_id = "mutuelle-session-1"

def ask(question):
    response = rag_with_memory.invoke(
        {"question": question},
        config={"configurable": {"session_id": session_id}}
    )
    print(f"\n❓ {question}\n")
    print("💬 Réponse :\n", response)
    print("\n" + "="*80)

ask("Quels sont les grands axes stratégiques du rapport d'activité ?")
ask("Peux-tu résumer ces axes en mettant l’accent sur la transformation digitale ?")
ask("Comment ces axes stratégiques influencent-ils la relation sociétaire ?")



❓ Quels sont les grands axes stratégiques du rapport d'activité ?

💬 Réponse :
 Les grands axes stratégiques du rapport d'activité du Crédit Mutuel se déclinent en trois priorités principales :

1. **Renforcer la compétitivité européenne** : Cela implique la mobilisation de financements pour stimuler l'innovation et la croissance durable au sein des banques coopératives.

2. **Garantir un cadre réglementaire équilibré** : Le Crédit Mutuel s'engage à défendre un cadre adapté aux spécificités des banques coopératives, ce qui inclut la surveillance des évolutions législatives majeures.

3. **Surveiller les évolutions législatives** : Cela comprend l'engagement actif du Crédit Mutuel à adapter ses pratiques aux changements réglementaires pour rester compétitif et conforme.

Ces axes stratégiques s'accompagnent d'une évaluation continue des impacts, risques et opportunités, intégrant des thématiques variées telles que le changement climatique, la biodiversité, et la conduite des affaires, 

In [39]:
ask("Et résume-moi tout cela en une phrase, comme si tu parlais à un sociétaire.")



❓ Et résume-moi tout cela en une phrase, comme si tu parlais à un sociétaire.

💬 Réponse :
 Au Crédit Mutuel, chaque sociétaire a voix au chapitre grâce à notre engagement en faveur de la solidarité, de l'égalité et de la responsabilité sociale, ce qui nous permet d'agir ensemble pour construire une banque plus proche de vos besoins et ancrée dans la communauté.

